# 🧠 Module 2 – Session 2 Assignment
# The Decoding Playbook

> **Goal:** Learn how an LLM engineer chooses decoding parameters for different business tasks.

---

## Before You Start

This assignment is **not** asking you to discover the mathematically best temperature.

Instead, imagine you joined a company and your manager asks:

> **"Which decoding settings should we use for each feature, and why?"**

Your job is to justify your engineering decisions with experiments.


# 📖 Story

You work at **Lumen Desk**.

The company has three AI products.

| Product | What the model should do |
|---|---|
| Ticket Tagger | Return ONE category only |
| Reply Drafter | Write a professional reply |
| Campaign Brainstormer | Generate creative marketing ideas |

Notice that these products have **different business goals**, so they probably need **different decoding strategies**.


# 🚀 Roadmap

You will repeat the same workflow three times.

```text
Understand the task
        ↓
Define success
        ↓
Design 3 configurations
        ↓
Run experiments
        ↓
Compare results
        ↓
Choose the winner
        ↓
Write your engineering recommendation
```


# Ticket Tagger

## Step 1 — Understand the Business Problem

### Success Criteria

**Exactly one category, deterministic, strict format.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical? **Yes.**
2. Is creativity helpful or harmful? **Creativity is harmful because the model should always produce the same category for the same input.**
3. Should the model take risks? **No. The model should return the most accurate and deterministic category without guessing.**

---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---:|---|---|
|**Primary**|0.0|20|1.0|0.0|Greedy|Provides deterministic and consistent classification with no randomness.|
|**Challenger A**|0.2|20|0.9|0.0|Sampling|Allows slight randomness while keeping the output stable and accurate.|
|**Challenger B**|0.4|10|0.8|0.0|Sampling|Tests whether increased randomness affects classification consistency.|

---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
You work as a customer support ticket classifier.

Read the customer's message and classify it into one category only.

Categories:
- Billing
- Technical Support
- Account
- Shipping

Customer Message:
"Hi, I ordered my laptop a week ago, but I still haven't received it. Can you check where my order is?"

Return only the category.
```

In [1]:
# Setup — run this cell FIRST (before the break, ideally)
%pip install -q torch transformers tiktoken matplotlib numpy

import numpy as np
import torch
import matplotlib.pyplot as plt
import tiktoken
from transformers import GPT2LMHeadModel, GPT2Tokenizer

torch.manual_seed(42)
np.random.seed(42)

tok = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
print("Setup OK — GPT-2 loaded,", sum(p.numel() for p in model.parameters())//1_000_000, "M parameters")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Setup OK — GPT-2 loaded, 124 M parameters


In [9]:
# ================================
# Run your experiments here
# ================================

prompt = """
You work as a customer support ticket classifier.

Read the customer's message and classify it into one category only.

Categories:
- Billing
- Technical Support
- Account
- Shipping

Customer Message:
"Hi, I ordered my laptop a week ago, but I still haven't received it. Can you check where my order is?"

Return only the category.
"""

inputs = tok(prompt, return_tensors="pt")


def run_generation(name, temperature, top_k, top_p, do_sample, runs):
    print("=" * 70)
    print(name)
    print("=" * 70)

    for i in range(runs):
        output = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=tok.eos_token_id
        )

        text = tok.decode(output[0], skip_special_tokens=True)

        print(f"\nRun {i+1}")
        print("-" * 30)
        print(text)
        print()


# ================================
# Primary (3 Runs)
# ================================
run_generation(
    name="Primary",
    temperature=0.0,
    top_k=20,
    top_p=1.0,
    do_sample=False,
    runs=3
)

# ================================
# Challenger A (3 Runs)
# ================================
run_generation(
    name="Challenger A",
    temperature=0.2,
    top_k=20,
    top_p=0.9,
    do_sample=True,
    runs=3
)

# ================================
# Challenger B (3 Runs)
# ================================
run_generation(
    name="Challenger B",
    temperature=0.4,
    top_k=10,
    top_p=0.8,
    do_sample=True,
    runs=3
)

Primary

Run 1
------------------------------

You work as a customer support ticket classifier.

Read the customer's message and classify it into one category only.

Categories:
- Billing
- Technical Support
- Account
- Shipping

Customer Message:
"Hi, I ordered my laptop a week ago, but I still haven't received it. Can you check where my order is?"

Return only the category.

Categories:

- Customer Service



Run 2
------------------------------

You work as a customer support ticket classifier.

Read the customer's message and classify it into one category only.

Categories:
- Billing
- Technical Support
- Account
- Shipping

Customer Message:
"Hi, I ordered my laptop a week ago, but I still haven't received it. Can you check where my order is?"

Return only the category.

Categories:

- Customer Service



Run 3
------------------------------

You work as a customer support ticket classifier.

Read the customer's message and classify it into one category only.

Categories:
- Billi

---

## Step 4 — Evaluate

Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary|Score|Observation|
|---|---:|---|---:|---|
|Primary|1|Returned **Customer Service** instead of one of the allowed categories.|7/10|Consistent, but ignored the required category list.|
|Primary|2|Returned **Customer Service** again.|7/10|Very consistent across runs.|
|Primary|3|Returned **Customer Service** again.|7/10|Deterministic behaviour.|
|Challenger A|1|Returned **Customer Support**.|6/10|Different wording, not in the allowed categories.|
|Challenger A|2|Returned **Customer Service**.|6/10|Slight variation due to sampling.|
|Challenger A|3|Returned **Business**.|5/10|Output became less relevant.|
|Challenger B|1|Returned **Business**.|4/10|High randomness reduced quality.|
|Challenger B|2|Returned **Customer Service**.|5/10|Inconsistent with previous run.|
|Challenger B|3|Returned **Customer Service**.|5/10|Still inconsistent and outside the allowed categories.|

### Questions

- **Which configuration won?**

  **Primary** performed the best because it produced the most consistent outputs across all runs.

- **Why?**

  The low temperature and greedy decoding made the model deterministic, resulting in stable outputs even though the category was not one of the allowed options.

- **Did the evidence surprise you?**

  No. Lower temperature produced more consistent outputs, while higher temperatures introduced more variation and randomness.

# Reply Drafter

## Step 1 — Understand the Business Problem

### Success Criteria

**Professional, coherent, polite, natural.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical?
   **No.**

2. Is creativity helpful or harmful?
   **Creativity is helpful, but it should be moderate to keep the reply professional and natural.**

3. Should the model take risks?
   **No. The model should provide polite, clear, and professional responses without making assumptions or giving incorrect information.**



---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---:|---|---|
|**Primary**|0.6|50|0.9|0.0|Sampling|Produces professional, natural, and balanced replies with moderate creativity.|
|**Challenger A**|0.4|40|0.8|0.0|Sampling|More conservative and consistent responses with less creativity.|
|**Challenger B**|0.8|60|1.0|0.0|Sampling|More creative and varied responses while remaining polite and coherent.|

---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
You are a customer support assistant.

Write a polite and professional reply to this customer.

Customer Message:
"My order is three days late. I'm disappointed because I needed it for an event. Can you please help?"

```

In [17]:
# ================================
# Run your experiments here
# ================================

prompt = """
You are a customer support assistant.

Write a polite and professional reply to this customer.

Customer Message:
"My order is three days late. I'm disappointed because I needed it for an event. Can you please help?"

"""

inputs = tok(prompt, return_tensors="pt")


def run_generation(name, temperature, top_k, top_p, do_sample, runs):
    print("=" * 70)
    print(name)
    print("=" * 70)

    for i in range(runs):

        output = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=tok.eos_token_id
        )

        # Remove the prompt from the output
        prompt_length = inputs["input_ids"].shape[1]
        generated_tokens = output[0][prompt_length:]

        response = tok.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        print(f"\nRun {i+1}")
        print("-" * 30)
        print(response)
        print()


# ======================================
# Primary
# ======================================
run_generation(
    name="Primary",
    temperature=0.6,
    top_k=50,
    top_p=0.9,
    do_sample=True,
    runs=3
)

# ======================================
# Challenger A
# ======================================
run_generation(
    name="Challenger A",
    temperature=0.4,
    top_k=40,
    top_p=0.8,
    do_sample=True,
    runs=3
)

# ======================================
# Challenger B
# ======================================
run_generation(
    name="Challenger B",
    temperature=0.8,
    top_k=60,
    top_p=1.0,
    do_sample=True,
    runs=3
)

Primary

Run 1
------------------------------
Customer Response:

"Thank you for your time."

Customer Response:

"I'm sorry for the delay. I have no idea what to do with this order. I'm sorry for the inconvenience. I can't order more than a few items at a time."


Customer Response:

"Thank you for your time. I appreciate your patience. I will


Run 2
------------------------------
You are a customer support assistant.

Write a polite and professional reply to this customer.

Customer Message:

"I am so sorry for the inconvenience. I would like to help you with your order. I am in need of a shower and it is very late. I will try to contact you later."


You are a customer support assistant.

Write a


Run 3
------------------------------
"Thank you for the order. I'll be back soon."

Customer Message:

"I ordered the same night and received the same day. I'm sorry for the inconvenience. I would like to know if the item is still available or if it is available again. Thank you."


"Tha

---

## Step 4 — Evaluate

Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary|Score|Observation|
|---|---:|---|---:|---|
|Primary|1|Started with a polite response but continued generating unrelated customer conversations.|7/10|Relevant at the beginning, but lost focus and generated extra unnecessary text.|
|Primary|2|Repeated the prompt and mixed it with a new customer message.|6/10|Partially followed the task but failed to produce a clean reply.|
|Primary|3|Generated a short reply, then switched back to customer messages.|6/10|Reasonably relevant at first, but drifted away from the requested format.|
|Challenger A|1|Repeated apology sentences many times.|4/10|Highly repetitive and lacked coherence.|
|Challenger A|2|Repeated customer messages and apologies instead of writing one reply.|4/10|Output became repetitive and unfocused.|
|Challenger A|3|Repeated the same apology and order problem multiple times.|3/10|Strong repetition reduced the quality of the response.|
|Challenger B|1|Generated a short but unrelated customer response.|5/10|More diverse, but not focused on the requested reply.|
|Challenger B|2|Switched to a completely different writing task.|2/10|Ignored the original task and generated irrelevant content.|
|Challenger B|3|Generated unrelated customer conversation and explanations.|3/10|High randomness caused the model to drift away from the prompt.|

### Questions

- **Which configuration won?**

  **Primary** performed the best overall.

- **Why?**

  It produced the most coherent and consistent outputs. Although the responses were still imperfect, they stayed closer to the requested customer support reply than the challenger configurations.

- **Did the evidence surprise you?**

  **No.** GPT-2 is a text completion model rather than an instruction-following model. The primary configuration produced more stable outputs, while the higher randomness in Challenger B caused the model to drift further from the requested task.

# Campaign Brainstormer

## Step 1 — Understand the Business Problem

### Success Criteria

**Creative, diverse, non-repetitive.**

### Reflection

Answer briefly before moving on:

1. Should the output always be identical?
   **No.**

2. Is creativity helpful or harmful?
   **Creativity is helpful because the goal is to generate diverse and original campaign ideas.**

3. Should the model take risks?
   **Yes, but only moderate risks. It should generate creative ideas while keeping them relevant and appropriate.**



---

## Step 2 — Design Three Configurations

You are **not** searching for the correct answer.

Instead, design:

- **Primary** → Your recommendation
- **Challenger A** → A reasonable alternative
- **Challenger B** → Another reasonable alternative

|Configuration|Temperature|Top-k|Top-p|Penalty|Greedy/Sampling|Why?|
|---|---:|---:|---:|---:|---|---|
|**Primary**|0.9|50|0.95|1.0|Sampling|Produces creative, diverse, and engaging campaign ideas while maintaining coherence.|
|**Challenger A**|0.9|50|0.95|1.2|Sampling|Applies a repetition penalty to reduce repeated words and encourage more varied ideas while keeping the same creativity settings.|
|**Challenger B**|1.1|80|1.0|1.0|Sampling|Encourages maximum creativity and diversity, even if some ideas become less focused.|

---

## Step 3 — Create One Representative Prompt

Choose ONE prompt that realistically represents this product.

```text
You are a marketing specialist.

Create 5 creative campaign ideas for launching a new healthy energy drink.


```

In [23]:
# ================================
# Run your experiments here
# ================================

prompt = """
You are a marketing specialist.

Create 5 creative campaign ideas for launching a new healthy energy drink.


"""

inputs = tok(prompt, return_tensors="pt")


def run_generation(name, temperature, top_k, top_p, penalty, do_sample, runs):
    print("=" * 70)
    print(name)
    print("=" * 70)

    for i in range(runs):

        output = model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=penalty,
            do_sample=do_sample,
            pad_token_id=tok.eos_token_id
        )

        # Remove the prompt
        prompt_length = inputs["input_ids"].shape[1]
        generated_tokens = output[0][prompt_length:]

        response = tok.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        print(f"\nRun {i+1}")
        print("-" * 30)
        print(response)
        print()


# ======================================
# Primary
# ======================================
run_generation(
    name="Primary",
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    penalty=1.0,
    do_sample=True,
    runs=3
)

# ======================================
# Challenger A (Repetition Penalty)
# ======================================
run_generation(
    name="Challenger A",
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    penalty=1.2,
    do_sample=True,
    runs=3
)

# ======================================
# Challenger B
# ======================================
run_generation(
    name="Challenger B",
    temperature=1.1,
    top_k=80,
    top_p=1.0,
    penalty=1.0,
    do_sample=True,
    runs=3
)

Primary

Run 1
------------------------------
The more we work together, the more products we can develop.

With you, we'll be able to push that energy-saving innovation forward as quickly as we can.


So, how do you bring your ideas to market?

Your business may be one of the world's top health-conscious businesses, but it's also one of the largest and most successful health insurance companies.

To make your company successful, you need to bring energy-saving ideas to market to ensure the benefits don't just come in a glass of water.

If it's a health


Run 2
------------------------------
This will be easy.

Don't create a list of 7 "great" food and beverage ideas you love.


Create 3 new ideas about your favourite meal suggestions.

Create 2 new "good" ideas about what to eat next.

Create 2 new ideas about your favourite food and drink ideas.


Make this list.

What you are doing is working to raise money for healthy energy drinks.

What you are doing is working to raise money for

---

## Step 4 — Evaluate

Score the behaviour, not factual correctness.

|Configuration|Run|Output Summary|Score|Observation|
|---|---:|---|---:|---|
|Primary|1|Generated marketing-related ideas with some creative language but drifted into business advice.|8/10|Creative and relevant at first, but became less focused on campaign ideas.|
|Primary|2|Produced brainstorming-style suggestions related to healthy drinks and marketing.|8/10|Reasonably creative and relevant, although the structure was inconsistent.|
|Primary|3|Generated motivational marketing advice instead of distinct campaign ideas.|7/10|Creative but less aligned with the requested output format.|
|Challenger A|1|Generated a website link and unrelated health content.|4/10|Lost focus and produced irrelevant promotional text.|
|Challenger A|2|Generated unrelated health and recipe recommendations.|4/10|Output was coherent but unrelated to campaign brainstorming.|
|Challenger A|3|Generated promotional sales text with unrealistic numbers.|5/10|Creative but not suitable as campaign ideas.|
|Challenger B|1|Generated marketing-related instructions and promotional content.|6/10|More diverse, but lacked clear campaign ideas.|
|Challenger B|2|Produced a mixed list of product suggestions and marketing advice.|5/10|Creative but inconsistent and partially unrelated.|
|Challenger B|3|Generated a long startup and fundraising discussion.|5/10|Highly creative but drifted away from the requested brainstorming task.|

### Questions

- **Which configuration won?**

  **Primary** performed the best overall.

- **Why?**

  It generated the most relevant and creative marketing-related content. Although it occasionally drifted away from the requested format, its outputs stayed closer to campaign brainstorming than the challenger configurations.

- **Did the evidence surprise you?**

  **No.** GPT-2 is a text completion model, so increasing randomness produced more diverse ideas but also caused the model to drift away from the intended task. The primary configuration achieved the best balance between creativity and coherence.

# 📝 Part C — The Decoding Playbook

Imagine a new engineer joins your team tomorrow.

They should be able to use this page **without reading the rest of the notebook.**

## Final Recommendations

|Feature|Recommended Configuration|Reason|
|---|---|---|
|Ticket Tagger|Temperature = 0.0, Greedy Decoding|Produces deterministic and consistent classifications with the same output every time.|
|Reply Drafter|Temperature = 0.6, Top-k = 50, Top-p = 0.9, Sampling|Generates polite, natural, and professional replies while maintaining consistency.|
|Campaign Brainstormer|Temperature = 0.9, Top-k = 50, Top-p = 0.95, Sampling|Produces creative, diverse, and engaging campaign ideas without becoming overly random.|

---

## House Rule #1

> Use greedy or low-temperature decoding for tasks that require consistent, repeatable, and structured outputs.

---

## House Rule #2

> Increase temperature and use sampling only when creativity and diversity improve the quality of the final result.

---

## Biggest Limitation

**GPT-2 is a small model.**

It is a text completion model with limited instruction-following ability. During the experiments, it often repeated text, drifted away from the prompt, or generated irrelevant content, which affected the quality and consistency of the outputs.

# ✅ Submission Checklist

- [ ] I defined success criteria before testing.
- [ ] I designed three configurations for every feature.
- [ ] Every stochastic configuration was executed at least three times.
- [ ] I scored outputs before deciding the winner.
- [ ] My final playbook is self-contained.
- [ ] All notebook cells are executed.

---

## ⭐ Bonus (Optional)

Complete ONE:

- Compare Greedy vs Sampling visually.
- Plot the effect of different temperatures.
- Demonstrate reproducibility using random seeds.
- Show a challenger configuration outperforming your primary recommendation.
